## 유사도 판단

In [10]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 파일, 필드명, userid
df = pd.read_feather('../output/lowdim_vectors(BERT ver.)(UI+UR)/Cameras_lowdim.feather')
vector = 'lowdim_vector'     
user1 = '5399214'
user2 = '5840262'

# userid와 vector 열을 추출
user_vectors = df[['UserID', vector]]

# 벡터를 리스트 형태로 변환 (예: 벡터는 문자열로 되어 있을 경우, 이를 리스트로 변환)
def str_to_vector(vector_str):
    if isinstance(vector_str, np.ndarray):
        return vector_str
    # 벡터 문자열을 쉼표로 구분하여 numpy 배열로 변환
    return np.array([float(x) for x in vector_str.strip('[]').split(' ')])

# 벡터 열을 numpy 배열로 변환
user_vectors[vector] = user_vectors[vector].apply(str_to_vector)

# 두 사용자(userid) 간 벡터 유사도 계산 함수
def calculate_cosine_similarity(user1, user2):
    if user1 not in user_vectors['UserID'].values or user2 not in user_vectors['UserID'].values:
        raise ValueError(f"One or both user IDs ({user1}, {user2}) not found.")
    
    # 두 사용자의 벡터를 가져옴
    vector1 = user_vectors[user_vectors['UserID'] == user1][vector].values[0]
    vector2 = user_vectors[user_vectors['UserID'] == user2][vector].values[0]
    
    # 코사인 유사도 계산
    similarity = cosine_similarity([vector1], [vector2])
    
    return similarity[0][0]


similarity = calculate_cosine_similarity(user1, user2)

print(f'{vector} - User {user1}와 User {user2} 간의 코사인 유사도: {similarity}')


lowdim_vector - User 5399214와 User 5840262 간의 코사인 유사도: 0.9549968117338374


## 연결확인

In [15]:
import csv

trustnetwork_file = '../dataset/trustnetwork.csv'

# 이제 CSV 파일을 다시 읽어서 set으로 변환
loaded_user_dict = {}

with open(trustnetwork_file, mode='r') as file:
    reader = csv.reader(file)
    next(reader)  # 헤더를 넘김
    
    for row in reader:
        user_id = row[0]
        trustors = set(row[1].split(','))  # 쉼표로 분리하여 set으로 변환
        loaded_user_dict[user_id] = trustors


#확인
check = 1
test = loaded_user_dict[user1]
if user2 in test:
    print("user1->user2")
    check = 0

test = loaded_user_dict[user2]
if user1 in test:
    print("user2->user1")
    check = 0

if check:
    print('non link')


non link


## 모든 user 유사도 판단

In [1]:
import pandas as pd
import numpy as np
import csv
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

# ================================
# Load trust network
# ================================
trustnetwork_file = '../dataset/trustnetwork.csv'
trust_dict = {}

with open(trustnetwork_file, mode='r') as file:
    reader = csv.reader(file)
    next(reader)
    for row in reader:
        user_id = int(row[0])
        trustors = set(map(int, row[1].split(',')))
        trust_dict[user_id] = trustors

# ================================
# Load lowdim vectors
# ================================
df = pd.read_feather('../codes2/Games_lowdim.feather')
df['UserID'] = df['UserID'].astype(int)

def str_to_vector(vec):
    if isinstance(vec, np.ndarray):
        return vec
    return np.array([float(x) for x in vec.replace('[', '').replace(']', '').split()])

df['lowdim_vector'] = df['lowdim_vector'].apply(str_to_vector)
user_vector_dict = dict(zip(df['UserID'], df['lowdim_vector']))
all_user_ids = list(user_vector_dict.keys())

# ================================
# Generate positive pairs
# ================================
positive_pairs = set()
for u in trust_dict:
    for v in trust_dict[u]:
        if u in user_vector_dict and v in user_vector_dict:
            positive_pairs.add(tuple(sorted([u, v])))

positive_pairs = list(positive_pairs)

# ================================
# Loop over different NEG_POS_RATIO values
# ================================
for NEG_POS_RATIO in range(1, 6):
    print(f"\n=== Evaluation for NEG_POS_RATIO = {NEG_POS_RATIO} ===")

    # Negative pair sampling
    negative_pairs = set()
    rng = np.random.default_rng(seed=42 + NEG_POS_RATIO)  # Different seed for each ratio
    target_neg_size = int(len(positive_pairs) * NEG_POS_RATIO)

    while len(negative_pairs) < target_neg_size:
        u1, u2 = rng.choice(all_user_ids, 2, replace=False)
        pair = tuple(sorted([u1, u2]))
        if pair in positive_pairs or pair in negative_pairs:
            continue
        negative_pairs.add(pair)

    # Evaluation dataset
    eval_pairs = positive_pairs + list(negative_pairs)
    labels = [1] * len(positive_pairs) + [0] * len(negative_pairs)

    # Compute similarity
    scores = []
    for u, v in eval_pairs:
        u_vec = user_vector_dict[u]
        v_vec = user_vector_dict[v]
        sim = cosine_similarity([u_vec], [v_vec])[0][0]
        scores.append(sim)

    # Evaluation
    thresholds = np.arange(0.0, 0.6, 0.1)
    print(f"{'Threshold':<10} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1 Score':<10} {'TP':<5} {'FP':<5} {'FN':<5} {'TN':<5}")

    for threshold in thresholds:
        y_pred = [1 if s >= threshold else 0 for s in scores]

        tp = sum(1 for p, l in zip(y_pred, labels) if p == 1 and l == 1)
        fp = sum(1 for p, l in zip(y_pred, labels) if p == 1 and l == 0)
        fn = sum(1 for p, l in zip(y_pred, labels) if p == 0 and l == 1)
        tn = sum(1 for p, l in zip(y_pred, labels) if p == 0 and l == 0)

        acc = round(accuracy_score(labels, y_pred) * 100, 2)
        prec = round(precision_score(labels, y_pred) * 100, 2)
        rec = round(recall_score(labels, y_pred) * 100, 2)
        f1 = round(f1_score(labels, y_pred)*100, 2)
        f1_2 = round(2 * (prec * rec) / (prec + rec), 2) if (prec + rec) > 0 else 0

        print(f"{threshold:<10.1f} {acc:<10.2f} {prec:<10.2f} {rec:<10.2f} {f1:<10.2f} {tp:<5} {fp:<5} {fn:<5} {tn:<5} {f1_2:<10.2f}")



=== Evaluation for NEG_POS_RATIO = 1 ===
Threshold  Accuracy   Precision  Recall     F1 Score   TP    FP    FN    TN   
0.0        50.49      50.27      91.44      64.87      18144 17951 1699  1892  64.87     
0.1        52.96      52.28      67.98      59.10      13490 12315 6353  7528  59.11     
0.2        54.89      57.90      35.88      44.30      7119  5177  12724 14666 44.30     
0.3        53.76      67.74      14.37      23.71      2851  1358  16992 18485 23.71     
0.4        51.80      75.76      5.31       9.92       1053  337   18790 19506 9.92      
0.5        50.66      82.71      1.66       3.26       330   69    19513 19774 3.25      

=== Evaluation for NEG_POS_RATIO = 2 ===
Threshold  Accuracy   Precision  Recall     F1 Score   TP    FP    FN    TN   
0.0        36.93      33.61      91.44      49.15      18144 35844 1699  3842  49.15     
0.1        47.72      35.26      67.98      46.44      13490 24766 6353  14920 46.44     
0.2        61.27      40.80      35.88

KeyboardInterrupt: 

## 파일 순회

In [8]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import csv
import glob
import os

trustnetwork_file = '../dataset/trustnetwork.csv'

output_folder = "../output"
os.makedirs(output_folder, exist_ok=True)

final_result_file = os.path.join(output_folder, "final_results/final_results(doc2vec ver.)+256dim")
feather_files = glob.glob("../output/lowdim_vectors/lowdim_vectors(doc2vec ver.)+256dim/*.feather")
vector_name = 'lowdim_vector'

# 신뢰 네트워크 로드
loaded_user_dict = {}
with open(trustnetwork_file, mode='r') as file:
    reader = csv.reader(file)
    next(reader)  # header skip
    for row in reader:
        user_id = row[0]
        trustors = set(row[1].split(','))
        loaded_user_dict[user_id] = trustors

# 벡터 문자열을 numpy array로 변환하는 함수
def str_to_vector(vector_str):
    if isinstance(vector_str, np.ndarray):  # 이미 numpy 배열이면 그대로 반환
        return vector_str
    return np.array([float(x) for x in vector_str.replace('[', '').replace(']', '').split()])

# 결과를 저장할 리스트 (CSV에 한 번에 저장할 용도)
result_list = []

# 각 feather 파일마다 처리
for feather_file in feather_files:
    print(f"Processing file: {feather_file}")
    
    # feather 파일 읽기
    df = pd.read_feather(feather_file)
    
    # lowdim_vector 전처리
    df[vector_name] = df[vector_name].apply(str_to_vector)
    
    # 사용자 리스트 및 벡터 배열 생성
    user_ids = np.array(df['UserID'])

    # 벡터들을 리스트로 수집하면서 길이 체크
    vector_list = []
    invalid_count = 0
    expected_dim = None

    for idx, vec in enumerate(df[vector_name]):
        try:
            vec_array = str_to_vector(vec)
            if expected_dim is None:
                expected_dim = len(vec_array)
            if len(vec_array) != expected_dim:
                print(f"Skipping row {idx} due to shape mismatch. Expected {expected_dim}, got {len(vec_array)}.")
                invalid_count += 1
                continue
            vector_list.append(vec_array)
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            invalid_count += 1

    # 벡터 스택 쌓기
    vectors = np.stack(vector_list)

    # 사용자 ID도 벡터 수만큼 필터링
    user_ids = user_ids[:len(vectors)]

    # 코사인 유사도 행렬 계산
    similarity_matrix = cosine_similarity(vectors)
    
    # 평가 변수 초기화
    error = 0
    true_count, false_count = 0, 0
    tp, fp, tn, fn = 0, 0, 0, 0
    
    num_users = len(user_ids)
    # 모든 사용자 쌍에 대해 계산 (중복 계산 방지)
    for i in range(num_users):
        for j in range(i + 1, num_users):
            user1, user2 = user_ids[i], user_ids[j]
            similarity = similarity_matrix[i, j]
            # 신뢰 관계 존재 여부 (양쪽 중 한쪽이라도 신뢰관계가 있으면 True)
            check = (user2 in loaded_user_dict.get(user1, set()) or 
                     user1 in loaded_user_dict.get(user2, set()))
            
            if ((similarity >= 0.5 and check) or (similarity < 0.5 and not check)):
                true_count += 1
            else:
                false_count += 1

            if similarity >= 0.5:
                if check:
                    tp += 1  # 실제 O, 예측 O
                else:
                    fp += 1  # 실제 X, 예측 O
            else:
                if check:
                    fn += 1  # 실제 O, 예측 X
                else:
                    tn += 1  # 실제 X, 예측 X
    
        # accuracy 계산 (퍼센트)
        accuracy = round((true_count / (true_count + false_count) * 100), 2) if (true_count + false_count) > 0 else None

        # precision, recall, F1 score 계산 (퍼센트 기준으로 변환)
        precision = round(tp / (tp + fp) * 100, 2) if (tp + fp) > 0 else 0
        recall = round(tp / (tp + fn) * 100, 2) if (tp + fn) > 0 else 0
        f1_score = round(2 * (precision * recall) / (precision + recall), 2) if (precision + recall) > 0 else 0

        
        # 결과 딕셔너리 생성
        result = {
            'file': os.path.basename(feather_file),
            'error': error,
            'true': true_count,
            'false': false_count,
            'accuracy': accuracy,
            'tp': tp,
            'fp': fp,
            'tn': tn,
            'fn': fn,
            'precision': precision,
            'recall': recall,
            'f1_score': f1_score
        }
    
    # 결과를 리스트에 추가
    result_list.append(result)
    
    print(f"Results saved for {feather_file}")

# 최종 결과를 하나의 CSV 파일로 저장
final_result_df = pd.DataFrame(result_list)
final_result_df.to_csv(final_result_file, index=False)

print(f"All results saved to {final_result_file}")


Processing file: ../output/lowdim_vectors/lowdim_vectors(doc2vec ver.)+256dim\Adult Products_lowdim.feather
Results saved for ../output/lowdim_vectors/lowdim_vectors(doc2vec ver.)+256dim\Adult Products_lowdim.feather
Processing file: ../output/lowdim_vectors/lowdim_vectors(doc2vec ver.)+256dim\Beauty_lowdim.feather
Results saved for ../output/lowdim_vectors/lowdim_vectors(doc2vec ver.)+256dim\Beauty_lowdim.feather
Processing file: ../output/lowdim_vectors/lowdim_vectors(doc2vec ver.)+256dim\Books_lowdim.feather
Results saved for ../output/lowdim_vectors/lowdim_vectors(doc2vec ver.)+256dim\Books_lowdim.feather
Processing file: ../output/lowdim_vectors/lowdim_vectors(doc2vec ver.)+256dim\Cameras_lowdim.feather
Results saved for ../output/lowdim_vectors/lowdim_vectors(doc2vec ver.)+256dim\Cameras_lowdim.feather
Processing file: ../output/lowdim_vectors/lowdim_vectors(doc2vec ver.)+256dim\Cars & Motorcycles_lowdim.feather
Results saved for ../output/lowdim_vectors/lowdim_vectors(doc2vec ve

In [9]:
df=  pd.read_csv(final_result_file)
print(final_result_file)
df

../output\final_results/final_results(doc2vec ver.)+256dim


,file,error,true,false,accuracy,tp,fp,tn,fn,precision,recall,f1_score
0,Adult Products_lowdim.feather,0,416,760,35.37,25,731,391,29,3.31,46.30,6.18
1,Beauty_lowdim.feather,0,6811407,83634,98.79,80,43114,6811327,40520,0.19,0.20,0.19
2,Books_lowdim.feather,0,6816594,100746,98.54,79,60248,6816515,40498,0.13,0.19,0.15
3,Cameras_lowdim.feather,0,1555745,18680,98.81,23,8933,1555722,9747,0.26,0.24,0.25
4,Cars & Motorcycles_lowdim.feather,0,1814108,16633,99.09,13,7370,1814095,9263,0.18,0.14,0.16
5,Ciao Caf__lowdim.feather,0,12936771,116724,99.11,173,53633,12936598,63091,0.32,0.27,0.29
6,Computers_lowdim.feather,0,3251978,23542,99.28,15,8145,3251963,15397,0.18,0.10,0.13
7,DVDs_lowdim.feather,0,12434523,299058,97.65,237,250849,12434286,48209,0.09,0.49,0.15
8,Education & Careers_lowdim.feather,0,551958,8253,98.53,32,3528,551926,4725,0.90,0.67,0.77
9,Electronics_lowdim.feather,0,2661121,21965,99.18,24,7642,2661097,14323,0.31,0.17,0.22


In [10]:
df=  pd.read_csv('../output/final_results/final_results(BERT ver.)(UI+UR)+norm+256dim')
df

,file,error,true,false,accuracy,tp,fp,tn,fn,precision,recall,f1_score
0,Adult Products_lowdim.feather,0,697,479,59.268707,19,444,678,35,0.041037,0.351852,0.073501
1,Beauty_lowdim.feather,0,6175987,719054,89.571433,8200,686654,6167787,32400,0.011801,0.201970,0.022299
2,Books_lowdim.feather,0,6298998,618342,91.060986,7966,585731,6291032,32611,0.013418,0.196318,0.025118
3,Cameras_lowdim.feather,0,716454,857971,45.505756,5062,853263,711392,4708,0.005898,0.518117,0.011662
4,Cars & Motorcycles_lowdim.feather,0,1041072,789669,56.866154,4026,784419,1037046,5250,0.005106,0.434023,0.010094
5,Ciao Caf_lowdim.feather,0,9906662,3146833,75.892793,32986,3116555,9873676,30278,0.010473,0.521402,0.020534
6,Computers_lowdim.feather,0,2917524,357996,89.070560,2517,345101,2915007,12895,0.007241,0.163314,0.013867
7,DVDs_lowdim.feather,0,11409279,1324302,89.599925,12572,1288428,11396707,35874,0.009663,0.259505,0.018633
8,Education & Careers_lowdim.feather,0,299689,260522,53.495736,2216,257981,297473,2541,0.008517,0.465840,0.016727
9,Electronics_lowdim.feather,0,2054082,629004,76.556696,3829,618486,2050253,10518,0.006153,0.266885,0.012028
